<a href="https://colab.research.google.com/github/Rahid-Khan/internship-work-/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rahid-Khan/internship-work-/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane (or freestyle) and why

**Lane: Lane 2 — Refresh / Content Opportunity Scoring.**

The question I'm chasing: *which pages should an editor review first for refresh, expansion, protection, pruning, or monitoring?* I picked this lane over the other three for three reasons:

1. It has an **observed target already sitting in the data**: `trend_direction` (computed from `trend_pct`, a real trailing measurement, not something I have to invent). Lane 1 and Lane 3 are descriptive/unsupervised, which is a fine place to start but doesn't hand me a target to score against; Lane 2 does.
2. It maps directly onto a **person and an action** — an SEO editor with limited hours per week deciding which of thousands of pages to open first. That's a ranking problem, and ranking problems are exactly where a model earns its keep over eyeballing a spreadsheet.
3. The repo's own reference pipeline (`scripts/01-05`) and example outputs (`outputs/model_report.md`, `outputs/refresh_queue_sample.csv`) are already built around this exact lane, which tells me the starter data was shaped to support it well — I'm not fighting the dataset's grain.

I'm treating this as **provisional**: I can confirm or swap by the end of Week 4 once I've done a real signal audit (ML-06) and know whether the association actually holds up.

In [4]:
# Section 1 support: confirm this lane's target actually exists in the starter data,
# and that it's an OBSERVED measurement, not something I'm defining after the fact.
import pandas as pd

df = pd.read_csv('/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv')
print('Rows, columns:', df.shape)
print()
print(df['trend_direction'].value_counts())


Rows, columns: (30000, 44)

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. The question: decision, action, cost of a wrong call

**Decision it improves:** which content items an editor should put at the top of this week's refresh queue, out of far more candidates than there's time to review by hand.

**Who acts, and what they do:** a content/SEO editor (or a small content team) opens the top of a ranked queue and, for each item, does one of: refresh/expand the content, protect it as-is, prune it, or just monitor it a while longer. The output isn't a black-box score alone — it comes with a reason code (e.g. "declining, has demand, thin word count") so the editor knows *why* it's near the top, not just that it is.

**Cost of a wrong call:**
- **False positive** (flagged as high-priority, isn't really): wasted editor hours on a page that was fine — the direct cost of the internship, since editor time is the scarce resource this whole project is meant to protect.
- **False negative** (a genuinely declining, high-demand page never surfaces near the top): a page keeps losing clicks/visibility for weeks before anyone notices — a slower, compounding cost, and worse for the client since it goes undetected rather than just being deprioritized.
- Because a false negative can compound silently while a false positive is merely wasteful, I'd rather bias the queue toward being over-inclusive near the top than under-inclusive.

**Why data or ML helps at all:** a single rule like "flag anything with `trend_direction == down`" already flags over half the dataset (see Section 3) — too blunt to be a priority queue. The real prioritization signal is the *combination* of decline, demand, staleness, and content depth, and how those signals interact isn't obvious enough to hand-write as a handful of if-statements. That tangle is exactly the case where a simple, explainable model (logistic regression / decision tree, per the repo's own pipeline) beats both a plain rule and a human skimming a spreadsheet.

In [5]:
# Section 2 support: show why a single blunt rule ("declining = review it") isn't a queue.
declining_share = (df['trend_direction'] == 'down').mean() * 100
print(f"Share of all rows already flagged by trend_direction == 'down': {declining_share:.1f}%")
print('That is over half the dataset -- far too many pages for an editor to actually work through.')


Share of all rows already flagged by trend_direction == 'down': 54.2%
That is over half the dataset -- far too many pages for an editor to actually work through.


## 3. Quick look at the data (2-3 real numbers)

Loaded `data/raw/content_refresh_anonymized.csv` above. The numbers below are why this lane looks worth the next 7 weeks:

In [6]:
# Three real numbers from the starter dataset that motivate Lane 2.
n = len(df)
n_clients = df['client_id'].nunique()
print(f'1) Base size: {n:,} content items across {n_clients} clients -- enough rows for a real train/holdout split by client.')
print()

declining = df['trend_direction'] == 'down'
has_demand = df['search_volume'] > 0
declining_with_demand = (declining & has_demand).mean() * 100
print(f"2) 'Declining AND has real search demand' (trend_direction == 'down' and search_volume > 0): {declining_with_demand:.1f}% of rows.")
print('   This is a MUCH smaller, more useful slice than the 54.2% flagged by decline alone --')
print('   it already shows that layering signals narrows a huge pool into something reviewable.')
print()

thin_declining_demand = (df['word_count_tier'].isin(['<1000', '1000-2000']) & declining & has_demand).mean() * 100
print(f'3) Thin content (< 2000 words) AND declining AND demand>0: {thin_declining_demand:.2f}% of rows.')
print('   A concrete, high-value review pocket: content that is both under-built and losing ground')
print('   while people are still searching for it -- the kind of page a refresh could plausibly fix.')
print()

# Bonus: reference the repo's own baseline-vs-model result, since it grounds why ML earns its
# place over a hand-rule in exactly this lane.
print('For reference, the repo\'s own reference pipeline reports Precision@50 rising from about')
print('0.24 (hand-written baseline rule) to about 0.74 (trained model) on this same dataset --')
print('a roughly 3x lift, which is the whole argument for why this lane is worth modeling at all.')


1) Base size: 30,000 content items across 32 clients -- enough rows for a real train/holdout split by client.

2) 'Declining AND has real search demand' (trend_direction == 'down' and search_volume > 0): 28.4% of rows.
   This is a MUCH smaller, more useful slice than the 54.2% flagged by decline alone --
   it already shows that layering signals narrows a huge pool into something reviewable.

3) Thin content (< 2000 words) AND declining AND demand>0: 4.88% of rows.
   A concrete, high-value review pocket: content that is both under-built and losing ground
   while people are still searching for it -- the kind of page a refresh could plausibly fix.

For reference, the repo's own reference pipeline reports Precision@50 rising from about
0.24 (hand-written baseline rule) to about 0.74 (trained model) on this same dataset --
a roughly 3x lift, which is the whole argument for why this lane is worth modeling at all.


## 4. Careful words: what I can and can't claim

**What I can claim, once this is built:**
- **Observed / measured** associations between content signals (staleness, word count, demand, past trend) and a subsequent decline, over the trailing windows present in the data.
- **Directional, decision-support** rankings: "these pages look like better first candidates for review than those" -- a prioritization aid, not a verdict.
- A measured **lift over a naive baseline** (e.g. precision@K of a ranked queue vs. a plain rule), computed on held-out, client-grouped data.

**What I will never claim:**
- That I predicted or reverse-engineered any part of Google's ranking algorithm. Everything here is observational correlation on FlyRank's own measured signals, not causal proof about search engine behavior.
- That flagging a page guarantees a refresh will help it -- I can say a page *looks like* a good candidate, not that fixing it will work.
- Any claim built on `trend_direction` or `trend_pct` used as a *feature* -- those two columns define the label itself, so using them as inputs would just be the model learning its own answer key (a leakage trap flagged directly in the data skill).
- Precision numbers computed on the same rows used to build the score -- any metric I report will come from a proper held-out, client-grouped split, never from data the ranking rule itself saw.

In [7]:
# Section 4 support: confirm the label-defining columns are not accidentally already in any
# feature list I might build later -- a quick self-reminder check, not a model yet.
leak_risk_cols = ['trend_direction', 'trend_pct']
print('Columns that define the target and must NEVER be used as features later:', leak_risk_cols)
print('Confirmed present in dataset:', all(c in df.columns for c in leak_risk_cols))


Columns that define the target and must NEVER be used as features later: ['trend_direction', 'trend_pct']
Confirmed present in dataset: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.